<a href="https://colab.research.google.com/github/Mehak1301/Credit-Card-Fraud-Detection/blob/main/credit_risk_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Credit Risk Modelling — PD / LGD / EAD / Expected Loss

End-to-end pipeline: data → feature engineering → WOE binning → logistic regression PD model → AUC/KS/Gini evaluation → LGD & EAD assumptions → portfolio Expected Loss (Basel II/III style).

In [3]:
!pip install -q kagglehub scikit-learn scipy pandas matplotlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve
from scipy.stats import ks_2samp

pd.set_option('display.max_columns', 100)
np.random.seed(42)

In [4]:
df = pd.read_csv("/content/credit_risk_dataset.csv")
print(df.shape)
df.head()

(32581, 12)


,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,22,59000,RENT,123.0,PERSONAL,D,35000,16.02,1,0.59,Y,3
1,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,0.10,N,2
2,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,0.57,N,3
3,23,65500,RENT,4.0,MEDICAL,C,35000,15.23,1,0.53,N,2
4,24,54400,RENT,8.0,MEDICAL,C,35000,14.27,1,0.55,Y,4


## EDA & Cleaning

Target: `loan_status` (1 = default/bad, 0 = fully paid/good).

In [5]:
df.columns = [c.lower().strip() for c in df.columns]
print(df.isna().sum())
print("\nDefault rate:", df['loan_status'].mean().round(4))

# clean known outliers
df = df[df['person_age'] <= 100]
if 'person_emp_length' in df.columns:
    df = df[df['person_emp_length'] <= 60]

# impute numeric NAs with median, categorical with mode
for col in df.select_dtypes(include=np.number).columns:
    df[col] = df[col].fillna(df[col].median())
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].fillna(df[col].mode()[0])

print("\nShape after cleaning:", df.shape)

person_age                       0
person_income                    0
person_home_ownership            0
person_emp_length              895
loan_intent                      0
loan_grade                       0
loan_amnt                        0
loan_int_rate                 3116
loan_status                      0
loan_percent_income              0
cb_person_default_on_file        0
cb_person_cred_hist_length       0
dtype: int64

Default rate: 0.2182

Shape after cleaning: (31679, 12)


## Feature Engineering

In [6]:
df['loan_to_income'] = df['loan_amnt'] / (df['person_income'] + 1)
df['income_per_year_employed'] = df['person_income'] / (df['person_emp_length'] + 1)

target = 'loan_status'
id_like_cols = []  # none in this dataset, kept for generality
feature_cols = [c for c in df.columns if c not in [target] + id_like_cols]

cat_cols = df[feature_cols].select_dtypes(include='object').columns.tolist()
num_cols = df[feature_cols].select_dtypes(include=np.number).columns.tolist()
print("Categorical:", cat_cols)
print("Numeric:", num_cols)

Categorical: ['person_home_ownership', 'loan_intent', 'loan_grade', 'cb_person_default_on_file']
Numeric: ['person_age', 'person_income', 'person_emp_length', 'loan_amnt', 'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length', 'loan_to_income', 'income_per_year_employed']


## Train/Test Split (before binning, to avoid leakage)

In [8]:
X = df[feature_cols].copy()
y = df[target].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
print(X_train.shape, X_test.shape, "train default rate:", y_train.mean().round(4))

(23759, 13) (7920, 13) train default rate: 0.2155


## WOE Binning + IV

WOE = ln(% good / % bad) in each bin — higher WOE means a safer bin. Numeric features are binned into quantiles; categorical features use their existing levels as bins.

In [9]:
def bin_feature(series, is_numeric, n_bins=8):
    if is_numeric:
        try:
            return pd.qcut(series, q=n_bins, duplicates='drop')
        except ValueError:
            return pd.qcut(series.rank(method='first'), q=n_bins, duplicates='drop')
    return series.astype(str)

def woe_iv_table(x_bin, y):
    tab = pd.DataFrame({'bin': x_bin, 'target': y})
    grp = tab.groupby('bin', observed=True)['target'].agg(['count', 'sum'])
    grp.columns = ['total', 'bad']
    grp['good'] = grp['total'] - grp['bad']
    grp['bad'] = grp['bad'].replace(0, 0.5)   # avoid log(0)
    grp['good'] = grp['good'].replace(0, 0.5)
    grp['dist_good'] = grp['good'] / grp['good'].sum()
    grp['dist_bad'] = grp['bad'] / grp['bad'].sum()
    grp['woe'] = np.log(grp['dist_good'] / grp['dist_bad'])
    grp['iv'] = (grp['dist_good'] - grp['dist_bad']) * grp['woe']
    return grp

iv_summary = {}
woe_maps = {}
bin_edges = {}

for col in feature_cols:
    is_num = col in num_cols
    x_bin_train = bin_feature(X_train[col], is_num)
    grp = woe_iv_table(x_bin_train, y_train)
    iv_summary[col] = grp['iv'].sum()
    woe_maps[col] = grp['woe'].to_dict()
    if is_num:
        bin_edges[col] = x_bin_train.cat.categories

iv_df = pd.Series(iv_summary).sort_values(ascending=False).rename('information_value')
print(iv_df.round(4))
print("\nRule of thumb: IV<0.02 useless, 0.02-0.1 weak, 0.1-0.3 medium, 0.3-0.5 strong, >0.5 suspicious/overfit")

loan_percent_income           0.9548
loan_to_income                0.9260
loan_grade                    0.8929
loan_int_rate                 0.6423
person_income                 0.4519
person_home_ownership         0.3763
cb_person_default_on_file     0.1644
loan_intent                   0.1049
loan_amnt                     0.0925
person_emp_length             0.0635
income_per_year_employed      0.0616
person_age                    0.0107
cb_person_cred_hist_length    0.0051
Name: information_value, dtype: float64

Rule of thumb: IV<0.02 useless, 0.02-0.1 weak, 0.1-0.3 medium, 0.3-0.5 strong, >0.5 suspicious/overfit


In [10]:
# Keep features with at least weak predictive power
selected_features = iv_df[iv_df >= 0.02].index.tolist()
print(f"Selected {len(selected_features)} / {len(feature_cols)} features:", selected_features)

def apply_woe(X_part, cols):
    out = pd.DataFrame(index=X_part.index)
    for col in cols:
        is_num = col in num_cols
        if is_num:
            binned = pd.cut(X_part[col], bins=[b.left for b in bin_edges[col]] + [bin_edges[col][-1].right],
                             include_lowest=True)
            binned = binned.astype(str)
        else:
            binned = X_part[col].astype(str)
        default_woe = np.mean(list(woe_maps[col].values()))
        out[col + '_woe'] = binned.map(lambda b: woe_maps[col].get(_match_key(b, woe_maps[col]), default_woe))
    return out

def _match_key(b, mapping):
    # exact match first, else fall back (handles interval string formatting edge cases)
    if b in mapping:
        return b
    for k in mapping:
        if str(k) == b:
            return k
    return None

X_train_woe = apply_woe(X_train, selected_features)
X_test_woe = apply_woe(X_test, selected_features)
X_train_woe.head()

Selected 11 / 13 features: ['loan_percent_income', 'loan_to_income', 'loan_grade', 'loan_int_rate', 'person_income', 'person_home_ownership', 'cb_person_default_on_file', 'loan_intent', 'loan_amnt', 'person_emp_length', 'income_per_year_employed']


,loan_percent_income_woe,loan_to_income_woe,loan_grade_woe,loan_int_rate_woe,person_income_woe,person_home_ownership_woe,cb_person_default_on_file_woe,loan_intent_woe,loan_amnt_woe,person_emp_length_woe,income_per_year_employed_woe
8602,-2.177823,-2.040558,0.353797,0.118636,0.131731,-0.496067,0.215708,-0.208959,0.301152,0.115414,0.102080
794,0.156562,0.157681,0.074884,-0.355150,0.961985,-0.496067,0.215708,-0.380548,-0.601729,0.115414,0.308195
11838,0.652819,0.617767,0.952716,0.784378,0.961985,-0.496067,0.215708,0.128430,0.117849,0.115414,0.049048
18574,-0.041694,-0.018829,0.353797,0.490284,0.412397,-0.496067,0.215708,-0.300589,-0.601729,0.133127,0.084383
27616,-2.177823,-2.040558,0.074884,0.195337,0.131731,-0.496067,0.215708,-0.208959,-0.273958,0.321124,0.017341


## Logistic Regression — PD Model

In [11]:
pd_model = LogisticRegression(max_iter=1000, class_weight='balanced')
pd_model.fit(X_train_woe, y_train)

train_pd = pd_model.predict_proba(X_train_woe)[:, 1]
test_pd = pd_model.predict_proba(X_test_woe)[:, 1]

coef_table = pd.DataFrame({
    'feature': X_train_woe.columns,
    'coefficient': pd_model.coef_[0]
}).sort_values('coefficient', ascending=False)
coef_table

,feature,coefficient
8,loan_amnt_woe,0.310325
3,loan_int_rate_woe,-0.054619
6,cb_person_default_on_file_woe,-0.061528
1,loan_to_income_woe,-0.383058
9,person_emp_length_woe,-0.525455
4,person_income_woe,-0.549332
0,loan_percent_income_woe,-0.769886
5,person_home_ownership_woe,-0.799000
10,income_per_year_employed_woe,-0.842691
2,loan_grade_woe,-1.153121


## Evaluation — AUC-ROC, KS Statistic, Gini Coefficient

In [ ]:
auc = roc_auc_score(y_test, test_pd)
gini = 2 * auc - 1
ks_stat = ks_2samp(test_pd[y_test == 1], test_pd[y_test == 0]).statistic

print(f"Test AUC-ROC     : {auc:.4f}")
print(f"Test Gini        : {gini:.4f}")
print(f"Test KS statistic: {ks_stat:.4f}  ({ks_stat*100:.1f}%)")

fpr, tpr, _ = roc_curve(y_test, test_pd)
plt.figure(figsize=(5,5))
plt.plot(fpr, tpr, label=f'PD model (AUC={auc:.3f})')
plt.plot([0,1],[0,1],'--', color='grey', label='Random')
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC Curve — PD Model'); plt.legend(); plt.show()

In [ ]:
# KS chart: cumulative % bad vs cumulative % good across score deciles
eval_df = pd.DataFrame({'pd': test_pd, 'y': y_test.values}).sort_values('pd', ascending=False).reset_index(drop=True)
eval_df['decile'] = pd.qcut(eval_df.index, 10, labels=False)
ks_table = eval_df.groupby('decile').agg(total=('y','count'), bad=('y','sum'))
ks_table['good'] = ks_table['total'] - ks_table['bad']
ks_table['cum_bad_pct'] = ks_table['bad'].cumsum() / ks_table['bad'].sum()
ks_table['cum_good_pct'] = ks_table['good'].cumsum() / ks_table['good'].sum()
ks_table['ks'] = (ks_table['cum_bad_pct'] - ks_table['cum_good_pct']).abs()
print(ks_table.round(3))
print("\nMax KS from decile table:", ks_table['ks'].max().round(4))

## LGD & EAD Assumptions

This dataset has no recovery/collections data, so LGD is *assumed* rather than modelled

In [12]:
test_portfolio = X_test.copy()
test_portfolio['pd'] = test_pd
test_portfolio['actual_default'] = y_test.values
test_portfolio['ead'] = test_portfolio['loan_amnt']

if 'loan_grade' in test_portfolio.columns:
    grade_lgd_map = {'A': 0.30, 'B': 0.35, 'C': 0.40, 'D': 0.45, 'E': 0.55, 'F': 0.65, 'G': 0.75}
    test_portfolio['lgd'] = test_portfolio['loan_grade'].map(grade_lgd_map).fillna(0.45)
else:
    test_portfolio['lgd'] = 0.45  # Basel II F-IRB flat retail unsecured assumption

test_portfolio['expected_loss'] = test_portfolio['pd'] * test_portfolio['lgd'] * test_portfolio['ead']
test_portfolio[['loan_amnt','pd','lgd','ead','expected_loss']].head()

,loan_amnt,pd,lgd,ead,expected_loss
17093,15000,0.824902,0.30,15000,3712.059483
30490,7500,0.596350,0.35,7500,1565.418469
17548,5000,0.910096,0.45,5000,2047.716507
21798,6000,0.119862,0.30,6000,215.751682
3606,10000,0.968352,0.40,10000,3873.407144


## Portfolio-Level Expected Loss

In [13]:
total_ead = test_portfolio['ead'].sum()
total_el = test_portfolio['expected_loss'].sum()
el_rate = total_el / total_ead

print(f"Portfolio EAD (test set)      : ${total_ead:,.0f}")
print(f"Portfolio Expected Loss (EL)  : ${total_el:,.0f}")
print(f"EL as % of exposure           : {el_rate*100:.2f}%")
print(f"Actual observed default rate  : {test_portfolio['actual_default'].mean()*100:.2f}%")

Portfolio EAD (test set)      : $76,944,675
Portfolio Expected Loss (EL)  : $12,946,798
EL as % of exposure           : 16.83%
Actual observed default rate  : 21.54%
